# Prophetチュートリアル第01回：基礎編：クイックスタート

## 1. Prophetの基礎と操作感（Python API）

Meta（旧Facebook）が開発した時系列予測ライブラリ **Prophet** の操作感は、Pythonの機械学習ライブラリ `scikit-learn` のモデルAPIと統一されています。

基本フローは非常にシンプルで、`Prophet` クラスのインスタンスを生成した後、過去データを `fit()` メソッドで学習させ、`predict()` メソッドで未来の予測値を算出します。

## 2. 入力データの形式（dsとyの制約）

Prophetに投入する入力データフレームには、**`ds`** と **`y`** という名前の2つのカラムが必須となります。

- **`ds`（datestamp）**: 日付または日時の識別カラム。Pandasが認識できる日付フォーマット（日付のみの場合は `YYYY-MM-DD`、時間を含む場合は `YYYY-MM-DD HH:MM:SS`）に揃える必要があります。
- **`y`**: 予測対象となる数値データ（売上高、アクセス数、需要量など）。

### サンプルデータセットの解説
本チュートリアルではサンプルとして、NFL（米アメフトリーグ）の名クォーターバックである **Peyton Manning（ペイトン・マニング）** のWikipediaページにおける日別ページビュー（PV）数の対数時系列データ（`example_wp_log_peyton_manning.csv`）を使用します。

このデータはアクセス数の急激なスパイクによる影響を和らげスケールを揃えるために対数変換（`log`）されています。また、複数スケールの季節性、トレンドの転換点、イベント効果（スーパーボウル出場日など）が含まれており、Prophetの機能を学ぶのに最適なサンプルです。

まず、必要なライブラリのインポートと前処理を行います。

In [ ]:
!pip install prophet
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from prophet import Prophet

In [ ]:
# サンプルデータの取得
df = pd.read_csv('https://raw.githubusercontent.com/facebook/prophet/main/examples/example_wp_log_peyton_manning.csv')
df.head()

## 3. モデルのインスタンス化と学習 (`fit`)

モデルオブジェクトを生成後、`fit()` メソッドに過去のデータフレーム `df` を渡すことで、モデルの学習（フィッティング）が実行されます。

In [ ]:
# モデルの定義とフィッティング
m = Prophet()
m.fit(df)

## 4. 未来の予測用データフレームの生成 (`make_future_dataframe`)

Prophetが提供するヘルパー関数 `make_future_dataframe()` を使用すると、指定した日数（`periods`）だけ未来に拡張された日付列を持つデータフレームを容易に生成できます。デフォルトでは過去の学習期間の日付も自動的に含まれるため、インサンプルでのあてはまり（適合度）も確認できます。

In [ ]:
# 今後365日分の未来の日付を含むデータフレームを生成
future = m.make_future_dataframe(periods=365)
future.tail()

## 5. 予測の実行と推論結果 (`predict`)

作成した `future` データフレームを `predict()` メソッドに渡すことで、各日付に対する予測処理（推論）を実行します。
算出される `forecast` オブジェクトには、主となる予測値 `yhat` や予測区間（`yhat_lower`, `yhat_upper`）、成分分解データが含まれます。

In [ ]:
# 予測（推論）の実行
forecast = m.predict(future)
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

## 6. 予測結果のプロットと可視化 (`plot`)

`m.plot()` メソッドに予測結果 `forecast` を渡すことで、過去の実測値（黒点）、予測トレンド、不確実性区間（青い帯）を可視化できます。

In [ ]:
# 予測結果のグラフ描画と画像保存
fig1 = m.plot(forecast)
fig1.savefig('01_plot_forecast.png', bbox_inches='tight')

## 7. 時系列の変動成分の分解可視化 (`plot_components`)

`m.plot_components()` メソッドを呼び出すことで、予測値を「全体トレンド」「曜日ごとの季節性」「年間の季節性」などの独立した成分に分解して視覚的に分析できます。

In [ ]:
# 分解成分のグラフ描画と画像保存
fig2 = m.plot_components(forecast)
fig2.savefig('01_plot_components.png', bbox_inches='tight')

## 生成画像のZip一括ダウンロード
以下のセルを実行すると、保存された画像（`01_plot_forecast.png`および`01_plot_components.png`）がZip形式で一括ダウンロードされます。

In [ ]:
import glob, zipfile
from google.colab import files

png_files = glob.glob('*.png')
with zipfile.ZipFile('images.zip', 'w') as zipf:
    for f in png_files:
        zipf.write(f)
files.download('images.zip')